# Multiscale 1D RM-CLEAN

Standard (Hogbom) RM-CLEAN models the Faraday dispersion function (FDF) as a
sum of delta functions. For a **Faraday-thick** source, whose FDF is genuinely
extended in Faraday depth, that becomes a picket fence of deltas that
under-represents the flux. Multiscale RM-CLEAN (Cornwell 2008; Offringa &
Smirnov 2017) cleans with extended kernels, so a thick component is modelled
as one broad feature.

Multiscale only helps when the structure is **resolved**, i.e. its Faraday
thickness $\Delta\phi$ exceeds the RMSF FWHM. We show three regimes on the
same broad, low-frequency (GMIMS-DRAGONS-like) coverage:

1. **Resolved thick** ($\Delta\phi \approx 5\times$ FWHM): multiscale wins.
2. **Marginally resolved** ($\Delta\phi \approx 1.5\times$ FWHM): multiscale
   starts to help.
3. **Unresolved / thin** ($\Delta\phi < $ FWHM): no benefit; matches
   single-scale at the default `scale_bias`, but an over-aggressive (low)
   `scale_bias` over-extends the point source.

The broad low band gives a fine RMSF (FWHM of a few rad m$^{-2}$), so a
moderate slab is genuinely thick.

In [ ]:
from __future__ import annotations

import logging

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from rm_lite.tools_1d import rmclean, rmsynth
from rm_lite.utils.logging import quiet_logs
from rm_lite.utils.synthesis import freq_to_lambda2, inverse_rmsynth_nufft

plt.rcParams["figure.dpi"] = 150
rng = np.random.default_rng(7)

# freq_arr_hz = np.linspace(0.3e9, 1.8e9, 500)
bw_low = 288
bw_mid = 144
bw_high = 288
low = np.linspace(943.5 - bw_low / 2, 943.5 + bw_low / 2, 36) * u.MHz
mid = np.linspace(1367.5 - bw_mid / 2, 1367.5 + bw_mid / 2, 9) * u.MHz
high = np.linspace(1655.5 - bw_high / 2, 1655.5 + bw_high / 2, 9) * u.MHz
freqs = np.concatenate([low, mid, high])
freq_arr_hz = freqs.to("Hz").value

lambda_sq = freq_to_lambda2(freq_arr_hz)
rms_noise = 0.05
frac_pol, psi0_deg, rm_radm2 = 0.5, 10.0, 25.0

## Helpers

`burn_slab` is a Burn slab of Faraday thickness $\Delta\phi$ ($\Delta\phi=0$ is
a thin source). `simulate` adds noise to $Q,U$; `clean_data` runs RM-synth then
single-scale and multiscale CLEAN; the plot helpers draw the input spectrum and
the clean FDFs.

In [ ]:
def burn_slab(
    lambda_sq_arr_m2: NDArray[np.float64], delta_rm_radm2: float
) -> NDArray[np.complex128]:
    """Burn slab P(lambda^2); delta_rm=0 is a Faraday-thin source."""
    return (
        frac_pol
        * np.exp(2j * (np.deg2rad(psi0_deg) + rm_radm2 * lambda_sq_arr_m2))
        * np.sinc(delta_rm_radm2 * lambda_sq_arr_m2 / np.pi)
    ).astype(np.complex128)


def simulate(delta_rm_radm2):
    model = burn_slab(lambda_sq, delta_rm_radm2)
    noise = rng.normal(0, rms_noise, freq_arr_hz.size) + 1j * rng.normal(
        0, rms_noise, freq_arr_hz.size
    )
    return (model + noise).astype(np.complex128)


def clean_data(complex_pol, scale_bias=None):
    # scale_bias=None uses the library default.
    complex_err = np.ones_like(complex_pol) * (rms_noise + 1j * rms_noise)
    with quiet_logs(logging.ERROR):
        synth = rmsynth.run_rmsynth(
            freq_arr_hz, complex_pol, complex_err, n_samples=20, do_fit_rmsf=True
        )
        single = rmclean.run_rmclean_from_synth(synth, auto_mask=10, auto_threshold=1)
        multi = rmclean.run_rmclean_from_synth(
            synth,
            auto_mask=10,
            auto_threshold=1,
            multiscale=True,
            scale_bias=scale_bias,
            max_iter_sub_minor=3000,
        )
    return synth, single, multi


def plot_input(complex_pol, delta_rm_radm2):
    # Model on a dense, uniform lambda^2 grid so the curve is smooth.
    dense_lambda_sq = np.linspace(lambda_sq.min(), lambda_sq.max(), 2000)
    model = burn_slab(dense_lambda_sq, delta_rm_radm2)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(
        lambda_sq,
        complex_pol.real,
        ".",
        ms=3,
        color="tab:red",
        label="Q (data)",
        alpha=0.2,
    )
    ax.plot(
        lambda_sq,
        complex_pol.imag,
        ".",
        ms=3,
        color="tab:blue",
        label="U (data)",
        alpha=0.2,
    )
    ax.plot(
        dense_lambda_sq, model.real, "-", lw=0.8, color="tab:red", label="Q (model)"
    )
    ax.plot(
        dense_lambda_sq, model.imag, "-", lw=0.8, color="tab:blue", label="U (model)"
    )
    ax.set(xlabel=r"$\lambda^2$ / m$^2$", ylabel="fractional polarisation")
    ax.legend()
    fig.tight_layout()


def plot_fdf(synth, single, multi, delta_rm_radm2, extra=None):
    phi = synth.fdf_arrs["phi_arr_radm2"].to_numpy().astype(float)
    dirty = synth.fdf_arrs["fdf_dirty_complex_arr"].to_numpy().astype(complex)
    s = single.fdf_arrs["fdf_clean_complex_arr"].to_numpy().astype(complex)
    m = multi.fdf_arrs["fdf_clean_complex_arr"].to_numpy().astype(complex)
    s_mod = single.fdf_arrs["fdf_model_complex_arr"].to_numpy().astype(complex)
    m_mod = multi.fdf_arrs["fdf_model_complex_arr"].to_numpy().astype(complex)
    extra_arr = (
        None
        if extra is None
        else (
            extra[0],
            extra[1].fdf_arrs["fdf_clean_complex_arr"].to_numpy().astype(complex),
            extra[1].fdf_arrs["fdf_model_complex_arr"].to_numpy().astype(complex),
        )
    )
    xlim = (rm_radm2 - 60, rm_radm2 + 60)
    if delta_rm_radm2 > 0:
        truth = np.where(np.abs(phi - rm_radm2) <= delta_rm_radm2 / 2, frac_pol, 0.0)

    # Restored clean FDF (model convolved with the clean beam).
    fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(6, 6))
    ax1.plot(phi, np.abs(dirty), color="0.7", label="dirty")
    ax1.plot(phi, np.abs(s), label="single-scale")
    ax1.plot(phi, np.abs(m), label="multiscale")
    if delta_rm_radm2 > 0:
        ax1.plot(phi, truth, "k--", lw=0.8, label="true slab")
    else:
        ax1.axvline(rm_radm2, ls="--", color="k", lw=0.8, label="true RM")
    if extra_arr is not None:
        ax1.plot(phi, np.abs(extra_arr[1]), label=extra_arr[0])
    ax1.set(xlabel=r"$\phi$ / rad m$^{-2}$", ylabel="|restored FDF|", xlim=xlim)
    ax1.legend()
    fig.tight_layout()

    # Clean-component model: single-scale is a picket fence of deltas,
    # multiscale is a few extended components.
    ax2.step(phi, np.abs(s_mod), label="single-scale model")
    ax2.step(phi, np.abs(m_mod), label="multiscale model")
    if extra_arr is not None:
        ax2.step(phi, np.abs(extra_arr[2]), label=extra_arr[0] + " model")
    ax2.set(xlabel=r"$\phi$ / rad m$^{-2}$", ylabel="|clean model|", xlim=xlim)
    ax2.legend()
    fig.tight_layout()


def summarise(name, single, multi):
    print(f"{name}: true flux {frac_pol}")
    print(
        f"  single-scale  mom0={float(single.fdf_parameters['mom0_debias'][0]):.3f}"
        f"  components={int(np.ravel(single.clean_parameters['n_iter'])[0])}"
    )
    print(
        f"  multiscale    mom0={float(multi.fdf_parameters['mom0_debias'][0]):.3f}"
        f"  components={int(np.ravel(multi.clean_parameters['n_iter'])[0])}"
    )

## Case 1: resolved Faraday-thick slab ($\Delta\phi = 30$, ~5x FWHM)

In [ ]:
pol1 = simulate(30.0)
plot_input(pol1, 30.0)

In [ ]:
synth1, single1, multi1 = clean_data(pol1)
fwhm = float(synth1.fdf_parameters["fwhm_rmsf_radm2"][0])
# rm-synth now returns the largest recoverable Faraday scale; multiscale uses it.
phi_max_scale = float(synth1.fdf_parameters["phi_max_scale_radm2"][0])
print(f"RMSF FWHM = {fwhm:.1f} rad/m^2  (slab = {30 / fwhm:.1f} x FWHM)")
print(
    f"max recoverable scale = {phi_max_scale:.0f} rad/m^2 "
    f"({phi_max_scale / fwhm:.1f} x FWHM)"
)
summarise("resolved thick", single1, multi1)
plot_fdf(synth1, single1, multi1, 30.0)

Single-scale scatters deltas across the slab; multiscale models it as one
extended feature, recovering more flux with far fewer components.

### Reverse RM-synthesis

Forward-model each clean model FDF back to $Q,U(\lambda^2)$ to check it
reproduces the data. Both track the data in band; note the **single-scale**
curve is rougher at high $\lambda^2$ (low SNR) because its many-delta model
fits the noise, whereas the compact multiscale model stays smooth. At
$\lambda^2=0$ (outside the band) the thick component is unconstrained, so the
curves diverge from the truth there.

In [ ]:
lam0 = float(synth1.fdf_parameters["lam_sq_0_m2"][0])
phi = synth1.fdf_arrs["phi_arr_radm2"].to_numpy().astype(float)
lambda_sq_ext = np.linspace(0.0, lambda_sq.max() * 1.05, 600)
model_ext = burn_slab(lambda_sq_ext, 30.0)

recon = {}
for label, res in (("single-scale", single1), ("multiscale", multi1)):
    fdf_model = res.fdf_arrs["fdf_model_complex_arr"].to_numpy().astype(complex)
    recon[label] = inverse_rmsynth_nufft(fdf_model, lambda_sq_ext, phi, lam0)

fig, axs = plt.subplots(3, 1, sharex=True, figsize=(6, 8))
for ax, comp, lab in zip(
    axs, (np.real, np.imag, np.abs), ("Q", "U", "|P|"), strict=True
):
    ax.axvspan(
        lambda_sq.min(), lambda_sq.max(), color="0.9", zorder=0, label="measured band"
    )
    ax.plot(lambda_sq, comp(pol1), ".", ms=3, color="0.4", label="data")
    ax.plot(lambda_sq_ext, comp(model_ext), "k--", lw=0.8, label="truth")
    for label in ("single-scale", "multiscale"):
        ax.plot(lambda_sq_ext, comp(recon[label]), lw=1, label=label)
    ax.set(ylabel=lab)
axs[-1].set_xlabel(r"$\lambda^2$ / m$^2$")
axs[0].legend(fontsize=7)
fig.tight_layout()

## Case 2: marginally resolved ($\Delta\phi = 10$, ~1.5x FWHM)

In [ ]:
pol2 = simulate(10.0)
plot_input(pol2, 10.0)

In [ ]:
synth2, single2, multi2 = clean_data(pol2)
summarise("marginally resolved", single2, multi2)
plot_fdf(synth2, single2, multi2, 10.0)

Near the resolution limit multiscale starts to pay off: a modest flux gain and
far fewer components, though the advantage is smaller than the fully resolved
case.

## Case 3: unresolved (thin) source ($\Delta\phi = 0$)

In [ ]:
pol3 = simulate(0.0)
plot_input(pol3, 0.0)

In [ ]:
synth3, single3, multi3 = clean_data(pol3)  # default scale_bias=0.8
synth3b, single3b, multi3b = clean_data(pol3, scale_bias=0.5)
summarise("thin, default scale_bias=0.8", single3, multi3)
summarise("thin, scale_bias=0.5 (too low)", single3b, multi3b)
plot_fdf(synth3, single3, multi3, 0.0, extra=("multiscale (scale_bias=0.5)", multi3b))

A point source has nothing extended to recover, so multiscale should match
single-scale. At the default `scale_bias=0.8` it does (same `mom0`). Pushing
`scale_bias` too low (0.5) makes it prefer extended kernels and **over-extend**
the point source (inflated `mom0`, broadened peak): a caution against an
over-aggressive scale bias for compact sources.

## Summary

| Regime | $\Delta\phi$ / FWHM | Multiscale vs single-scale |
|---|---|---|
| Resolved thick | $\gtrsim$ few | more flux, far fewer components |
| Marginal | $\sim 1$ | modest gain |
| Thin | $< 1$ | matches single-scale; over-extends if `scale_bias` too low |

Reach for multiscale when $\Delta\phi \gtrsim$ RMSF FWHM. Aggressiveness is set
by options, not code: `scale_bias` (lower = stronger large-scale preference),
`auto_threshold`, `auto_mask`. Enable with `multiscale=True` on
`run_rmclean_from_synth` (1D) or `rmclean_3d_from_synth` (3D).